# NB02 — Potsdam Quantitative Eval

**Runs on Kaggle T4 GPU.**

**Datasets to attach:**
- `dummyirl/sam3-weights` — SAM3 checkpoint
- `dummyirl/6isprs` — Potsdam ISPRS GeoTIFF + labels

**What this does:**
1. Installs env (conda + mmcv + mmseg, ~15 min)
2. Clones `HarishDeepak/SegEarth-OV-3` (our improved fork)
3. Stages Potsdam val tile `6_15` at the path `cfg_potsdam.py` expects
4. Runs `eval.py configs/cfg_potsdam.py` → prints mIoU / mAcc
5. Displays per-class IoU table

> Val tile is always `6_15` (tile-level split, no random split — patch overlap would leak).

 NB02 runs a quantitative evaluation of SegEarth-OV-3 on the Potsdam ISPRS dataset and gives you a real mIoU number.

    Specifically it:

    1. Clones your fork of SegEarth-OV-3 from GitHub (which has your enriched multi-synonym prompts)
    2. Loads the SAM3 checkpoint from /kaggle/input/sam3-weights/sam3.pt
      3. Runs inference on the val tile 6_15 (a 6000×6000 image) using sliding window
      4. Converts the RGB labels to indexed (0–5) since Potsdam labels are color-coded
      5. Computes mIoU and mAcc by comparing predictions against the ground truth label
    6. Saves results — the prediction PNG and metrics CSV to /kaggle/working/

    The goal is to get your own measured number to compare against the literature figure of 57.8% mIoU that SegEarth-OV-3 claims on Potsdam zero-shot.

## 1 — Environment setup (~15 min, run once per session)

In [ ]:
import os

!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda_installer.sh
!bash /tmp/miniconda_installer.sh -b -p /tmp/miniconda

os.environ.pop("PYTHONPATH", None)
os.environ["PATH"] = "/tmp/miniconda/bin:" + os.environ["PATH"]

!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
!conda --version

In [ ]:
!/tmp/miniconda/bin/conda create -n segearth python=3.10 -y

In [57]:
!conda run -n segearth pip install torch==2.4.0 torchvision==0.19.0 -q

In [58]:
!conda run -n segearth pip install openmim -q
!conda run -n segearth mim install "mmcv==2.2.0" -q
!conda run -n segearth pip install "mmsegmentation==1.2.2" -q

In [ ]:
!conda run -n segearth pip install openpyxl -q

In [ ]:
%%bash
# mmseg 1.2.2 declares mmcv<2.2.0 as max; patch so version check passes
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import pathlib
f = pathlib.Path("/tmp/miniconda/envs/segearth/lib/python3.10/site-packages/mmseg/__init__.py")
f.write_text(f.read_text().replace("MMCV_MAX = '2.2.0'", "MMCV_MAX = '2.3.0'"))
print("Patched MMCV_MAX → 2.3.0")
EOF
pip install numpy==1.26.4 -q

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
python - << 'EOF'
import mmcv; print("MMCV:", mmcv.__version__)
from mmcv.ops import point_sample; print("MMCV OPS OK")
from mmseg.structures import SegDataSample; print("MMSEG OK")
import torch; print("CUDA:", torch.cuda.is_available())
EOF

## 2 — Clone our fork

In [ ]:
import subprocess, os
from pathlib import Path

REPO = Path("/tmp/SegEarth-OV-3")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
    print(f"Updated → {REPO}")
else:
    subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/HarishDeepak/rg-segearth-ov3", str(REPO)],
        check=True)
    print(f"Cloned → {REPO}")

os.chdir(REPO)
!conda run -n segearth pip install -r requirements.txt -q

## 3 — Stage Potsdam val tile

`cfg_potsdam.py` expects:
```
/kaggle/working/PotsdamEval/
  img_dir/val/         ← *_RGB.tif  (val tile 6_15)
  ann_dir_indexed/val/ ← *_label_noBoundary.tif
```

In [62]:
from pathlib import Path

POTSDAM_INPUT = Path("/kaggle/input/datasets/dummyirl/6isprs")
EVAL_ROOT     = Path("/kaggle/working/PotsdamEval")
VAL_TILE      = "6_15"

img_dst   = EVAL_ROOT / "img_dir/val"
label_dst = EVAL_ROOT / "ann_dir_indexed/val"
img_dst.mkdir(parents=True, exist_ok=True)
label_dst.mkdir(parents=True, exist_ok=True)

img_files   = sorted(POTSDAM_INPUT.rglob(f"*{VAL_TILE}*_RGB.tif"))
label_files = sorted(POTSDAM_INPUT.rglob(f"*{VAL_TILE}*_label_noBoundary.tif"))

if not img_files:
    print("No val tile images found. Contents of Potsdam dataset:")
    for p in sorted(POTSDAM_INPUT.rglob("*.tif"))[:20]: print(" ", p.name)
else:
    for p in img_files:
        dst = img_dst / p.name
        if not dst.exists(): dst.symlink_to(p)
    for p in label_files:
        dst = label_dst / p.name
        if not dst.exists(): dst.symlink_to(p)
    print(f"Images:  {len(list(img_dst.iterdir()))}")
    print(f"Labels:  {len(list(label_dst.iterdir()))}")
    print("Staging done.")

Images:  1
Labels:  1
Staging done.


## 3b — Convert RGB labels → indexed (0–5)

`_label_noBoundary.tif` uses RGB colors, not class indices. This converts them so `eval.py` can read them.

In [ ]:
import numpy as np
from PIL import Image
from pathlib import Path

RGB_TO_IDX = {
    (255, 255, 255): 0,  # impervious surface
    (  0,   0, 255): 1,  # building
    (  0, 255, 255): 2,  # low vegetation
    (  0, 255,   0): 3,  # tree
    (255, 255,   0): 4,  # car
    (255,   0,   0): 5,  # clutter
}

label_dst = Path("/kaggle/working/PotsdamEval/ann_dir_indexed/val")

for lbl_path in sorted(label_dst.iterdir()):
    raw = np.array(Image.open(lbl_path))
    if raw.ndim == 2:
        # Already single-band indexed (values 0–5) — no conversion needed
        idx = raw.astype(np.uint8)
        print(f"Labels already indexed. Unique values: {np.unique(idx)}")
    else:
        rgb = raw[:, :, :3]
        idx = np.zeros(rgb.shape[:2], dtype=np.uint8)
        for color, cls in RGB_TO_IDX.items():
            mask = (rgb[:,:,0] == color[0]) & (rgb[:,:,1] == color[1]) & (rgb[:,:,2] == color[2])
            idx[mask] = cls
        print(f"Converted RGB → indexed. Unique class indices: {np.unique(idx)}")
    lbl_path.unlink()
    Image.fromarray(idx).save(lbl_path)

print(f"Done: {len(list(label_dst.iterdir()))} label(s) written.")

## 4 — Run quantitative eval

In [ ]:
%%bash
source /tmp/miniconda/bin/activate segearth
export MPLBACKEND=Agg  # must come after activate (activate.d scripts can reset it)
cd /tmp/SegEarth-OV-3
python eval.py configs/cfg_potsdam.py \
  --cfg-options \
  test_dataloader.dataset.data_root=/kaggle/working/PotsdamEval \
  model.slide_stride=512 \
  default_hooks.visualization.draw=True \
  default_hooks.visualization.interval=1 \
  visualizer.save_dir=/kaggle/working/show_dir/ \
  visualizer.alpha=1.0 \
  2>&1 | tee /kaggle/working/potsdam_eval_log.txt

cp /tmp/SegEarth-OV-3/work_dirs/cfg_potsdam/results.txt /kaggle/working/nb02_results.txt 2>/dev/null || true
cp /tmp/SegEarth-OV-3/results.xlsx /kaggle/working/nb02_results.xlsx 2>/dev/null || true

## 5 — Display results

In [65]:
import re
from pathlib import Path

log = Path("/kaggle/working/potsdam_eval_log.txt").read_text(errors="replace")

# Print last 50 lines (where MMSeg prints the metrics table)
lines = log.strip().splitlines()
print("\n".join(lines[-50:]))

        'mIoU',
        'mFscore',
    ], type='IoUMetric')
test_pipeline = [
    dict(type='LoadImageFromFile'),
    dict(type='LoadAnnotations'),
    dict(type='PackSegInputs'),
]
vis_backends = [
    dict(type='LocalVisBackend'),
]
visualizer = dict(
    alpha=0.7,
    name='visualizer',
    type='SegLocalVisualizer',
    vis_backends=[
        dict(type='LocalVisBackend'),
    ])
work_dir = './work_dirs/cfg_potsdam'

Traceback (most recent call last):
  File "/kaggle/working/SegEarth-OV-3/eval.py", line 347, in <module>
    main()
  File "/kaggle/working/SegEarth-OV-3/eval.py", line 290, in main
    .from_cfg(cfg)
  File "/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmengine/runner/runner.py", line 462, in from_cfg
    runner = cls(
  File "/kaggle/working/miniconda/envs/segearth/lib/python3.10/site-packages/mmengine/runner/runner.py", line 416, in __init__
    self.visualizer = self.build_visualizer(visualizer)
  File "/kaggle/working/miniconda/envs/segeart

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path

PALETTE = np.array([
    [255,255,255], [0,0,255], [0,255,255],
    [0,255,0], [255,255,0], [255,0,0],
], dtype=np.uint8)
CLASS_NAMES = ["impervious surface", "building", "low vegetation", "tree", "car", "clutter"]

img_path   = next(Path("/kaggle/working/PotsdamEval/img_dir/val").glob("*.tif"), None)
gt_path    = next(Path("/kaggle/working/PotsdamEval/ann_dir_indexed/val").glob("*.tif"), None)
show_dir   = Path("/kaggle/working/show_dir")
pred_files = sorted(show_dir.rglob("*.png")) if show_dir.exists() else []

if not pred_files:
    print("No prediction PNG found — check potsdam_eval_log.txt for errors")
elif img_path and gt_path:
    img = np.array(Image.open(img_path))[:,:,:3]
    gt  = np.array(Image.open(gt_path))
    raw = np.array(Image.open(pred_files[0]))

    # mmseg saves [GT | Prediction] side by side with alpha=1.0 (pure class colors).
    # Each panel is square (6000×6000 original), legend rows sit below the image content.
    # w_panel == image content height, so [:w_panel] crops out the legend.
    w_panel  = raw.shape[1] // 2
    pred_rgb = raw[:w_panel, w_panel:, :3]   # right half, image rows only
    gt_rgb   = PALETTE[gt.clip(0, 5)]

    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    for ax, arr, title in zip(axes,
                               [img, gt_rgb, pred_rgb],
                               ["RGB image", "Ground truth", "Prediction (baseline)"]):
        ax.imshow(arr)
        ax.set_title(title, fontsize=15, pad=8)
        ax.axis("off")

    legend_patches = [
        mpatches.Patch(color=np.array(PALETTE[i]) / 255., label=CLASS_NAMES[i])
        for i in range(len(CLASS_NAMES))
    ]
    fig.legend(handles=legend_patches, loc="lower center", ncol=6,
               fontsize=11, bbox_to_anchor=(0.5, -0.04), frameon=False)

    plt.tight_layout()
    out = Path("/kaggle/working/nb02_gt_vs_pred.png")
    plt.savefig(out, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out}")